# 🎨 ComfyUI Colab — FLUX.1-schnell GGUF Q5 (bản viết lại, tối ưu 2026-09)

Pipeline: **FLUX.1-schnell Q5_K_S (UNET GGUF)** + T5-XXL Q4_K_M + CLIP-L + VAE `ae` —
tổng ~12 GB model, chạy được trên **Colab free T4 16 GB**.

| Pipeline | Node | Việc nó làm | Thời gian/ảnh 1024² (T4) |
|---|---|---|---|
| `flux_q5_fast` | 9 | 4 bước, không detailer | ~15-20 s |
| `flux_q5_standard` | 13 | 4 bước + sửa mặt (YOLO+SAM) | ~25-35 s |
| `flux_q5_quality` | 15 | + sửa tay | ~40-55 s |
| `flux_q5_hires` | 17 | + upscale 1.5× rồi lấy lại chi tiết | ~70-90 s |
| `flux_q5_inpaint` | 12 | sửa vùng tô trên ảnh có sẵn | ~15-25 s |

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5, sau đó **Cell 6 (tạo ảnh ngay trong Colab)**
hoặc mở giao diện ComfyUI qua link cloudflared. Cell 7 = inpaint vẽ tay, Cell 8 = chẩn đoán.

### Bản này sửa gì so với bản cũ (kết quả quét 2026-09-24)
1. **Thiếu package `gguf`** → node `UnetLoaderGGUF` không load được → *mọi* workflow GGUF chết.
2. **Thiếu `scikit-image`** → Impact Pack raise ngay khi import → không có `FaceDetailer`/`SAMLoader`.
3. **Workflow cũ thiếu input required** `wildcard`, `cycle` của `FaceDetailer` → ComfyUI từ chối prompt.
4. **`UltralyticsDetectorProvider` thiếu tiền tố `bbox/`** → Impact Subpack không tìm ra file YOLO.
5. **`hf_hub_download(resume_download=True)`** đã bị bỏ từ `huggingface_hub` 1.x → mọi mirror HF lỗi `TypeError`.
6. **`--lowvram` không còn tác dụng** trên ComfyUI mới (Dynamic VRAM bật mặc định cho NVIDIA) → bỏ, dùng `--reserve-vram`.
7. Bỏ toàn bộ phần tìm/tải **WAI-illustrious (SDXL 6.9 GB)** — bản này chỉ chạy FLUX, không dùng tới.
8. Workflow được **nhúng thẳng trong notebook** (bản cũ tải từ repo `caone1196-sketch/t-i-li-u` — link chết).
9. Detailer giảm `guide_size 512→384`, `max_size 1024→768`, steps `6→4` → nhanh hơn ~35 % mà vẫn đủ nét.

Chi tiết bằng chứng: `docs/AUDIT_FLUX_2026-09.md`. Kiểm tra tĩnh workflow: `python3 scripts/validate_workflows.py`.


In [ ]:
# @title ⚙️ CELL 1 — GPU + Drive + ComfyUI
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_MODEL_DIR = ""  # @param {type:"string"}
COMFY_REF = "master"  # @param {type:"string"}
USE_HF_MIRROR = False  # @param {type:"boolean"}

import os, sys, json, time, subprocess

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

# ---------- 1) GPU ----------
gpu_name, vram_gb = 'CPU', 0.0
try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        text=True).strip().splitlines()[0]
    parts = [p.strip() for p in out.split(',')]
    gpu_name, vram_gb = parts[0], float(parts[1]) / 1024.0
except Exception as e:
    log(f'⚠️ Không thấy GPU ({e}) — đổi Runtime → Change runtime type → T4 GPU')
log(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')

if vram_gb >= 24:
    PROFILE = 'big'        # A100/L4-40: giữ model thường trú trong VRAM
elif vram_gb >= 12:
    PROFILE = 't4'         # T4/L4 16GB: cấu hình mặc định của notebook này
else:
    PROFILE = 'small'
log(f'PROFILE = {PROFILE}')

if USE_HF_MIRROR:
    os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
    log('HF_ENDPOINT = https://hf-mirror.com')
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

# ---------- 2) Drive (tuỳ chọn) ----------
drive_ok = False
if USE_DRIVE:
    log('Mount Drive (nếu treo >60s: Runtime → Interrupt, tick USE_DRIVE=False, chạy lại)')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_ok = True
        log('Drive OK')
    except Exception as e:
        log(f'Drive lỗi ({e}) → dùng /content (mất model khi tắt runtime)')

ROOT = None
if drive_ok:
    if DRIVE_MODEL_DIR.strip() and os.path.isdir(DRIVE_MODEL_DIR.strip()):
        ROOT = DRIVE_MODEL_DIR.strip()
    else:
        md = '/content/drive/MyDrive'
        cands = [os.path.join(md, n) for n in ('AI_Models', 'AI_models', 'ComfyUI_models')]
        try:
            sc = os.path.join(md, '.shortcut-targets-by-id')
            cands += [os.path.join(sc, n) for n in os.listdir(sc)]
        except Exception:
            pass
        for c in cands:
            if os.path.isdir(c) and (os.path.isdir(os.path.join(c, 'gguf'))
                                     or os.path.isdir(os.path.join(c, 'checkpoints'))):
                ROOT = c
                break
        ROOT = ROOT or os.path.join(md, 'AI_Models')
ROOT = ROOT or '/content/AI_Models'

DIRS = {
    'root': ROOT,
    'gguf': f'{ROOT}/gguf',                 # UNET GGUF  → models/unet
    'clip': f'{ROOT}/clip',                 # CLIP-L + T5 GGUF → models/clip
    'vae':  f'{ROOT}/vae',                  # ae.safetensors
    'yolo': f'{ROOT}/ultralytics/bbox',     # YOLO mặt/tay
    'sam':  f'{ROOT}/sams',                 # SAM ViT-B
    'upscale': f'{ROOT}/upscale_models',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
DIRS['profile'] = PROFILE
DIRS['gpu'] = gpu_name
DIRS['vram_gb'] = vram_gb
with open('/content/mode_ai_paths.json', 'w') as f:
    json.dump(DIRS, f, indent=1)
log(f'ROOT model = {ROOT}')
for k, v in DIRS.items():
    log(f'   {k:8} {v}')

# ---------- 3) ComfyUI ----------
COMFY = '/content/ComfyUI'
os.chdir('/content')
ref = (COMFY_REF or 'master').strip()
if os.path.isfile(f'{COMFY}/main.py'):
    log('ComfyUI đã có — bỏ qua clone (xoá /content/ComfyUI nếu muốn cài lại)')
else:
    log(f'Clone ComfyUI @ {ref}')
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', ref,
                        'https://github.com/comfyanonymous/ComfyUI', COMFY])
    if r.returncode != 0:
        log('--branch thất bại (có thể là commit SHA) → clone đầy đủ rồi checkout')
        subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI', COMFY], check=True)
        subprocess.run(['git', 'checkout', ref], cwd=COMFY, check=True)
    subprocess.run(['git', 'log', '-1', '--format=ComfyUI %h %cd %s', '--date=short'], cwd=COMFY)

# ---------- 4) requirements (KHÔNG đụng torch của Colab) ----------
os.chdir(COMFY)
subprocess.run("grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt "
               "> /content/req_notorch.txt", shell=True, check=True)
log('pip install requirements (bỏ torch/torchvision/torchaudio)')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
                '-r', '/content/req_notorch.txt'])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ torch không thấy CUDA — kiểm tra Runtime type là GPU (T4)')
log(f'PyTorch {getattr(torch, "__version__", "?")} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}')
log('✅ Xong Cell 1 → chạy Cell 2')


In [ ]:
# @title 🧩 CELL 2 — Custom node + dependency (bản đã sửa) + kiểm tra import
PIN_GGUF = "main"  # @param {type:"string"}
PIN_IMPACT = "main"  # @param {type:"string"}

import os, sys, re, json, time, subprocess

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

P = json.load(open('/content/mode_ai_paths.json'))
COMFY = '/content/ComfyUI'
CN = f'{COMFY}/custom_nodes'
os.makedirs(CN, exist_ok=True)

def clone(url, folder, ref='main'):
    path = os.path.join(CN, folder)
    if os.path.isdir(path) and os.listdir(path):
        log(f'{folder} đã có — bỏ qua')
        return
    log(f'Clone {folder} @ {ref}')
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', ref, url, path])
    if r.returncode != 0:
        subprocess.run(['git', 'clone', url, path], check=True)

clone('https://github.com/city96/ComfyUI-GGUF.git', 'ComfyUI-GGUF', PIN_GGUF)
clone('https://github.com/ltdrdata/ComfyUI-Impact-Pack.git', 'ComfyUI-Impact-Pack', PIN_IMPACT)
clone('https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git', 'ComfyUI-Impact-Subpack', PIN_IMPACT)

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-input', *args])

# (a) ComfyUI-GGUF khai báo: gguf>=0.13, sentencepiece, protobuf  ← bản cũ THIẾU, node GGUF chết
log('pip: gguf + sentencepiece + protobuf (ComfyUI-GGUF)')
pip('gguf>=0.13.0', 'sentencepiece', 'protobuf')

# (b) Impact Pack khai báo: scikit-image, piexif, dill, segment-anything, matplotlib, transformers
#     (opencv đã có sẵn trên Colab dưới dạng opencv-contrib-python nên KHÔNG cài lại)
log('pip: scikit-image + piexif + dill + segment-anything + matplotlib (Impact Pack)')
pip('scikit-image', 'piexif', 'dill', 'segment-anything', 'matplotlib')

# (c) Impact Subpack cần ultralytics. Cài --no-deps để KHÔNG kéo theo opencv/numpy mới
#     (tránh phá môi trường torch của Colab), rồi bù các package còn thiếu.
#     Đường chạy YOLO (inference) của ultralytics chỉ cần ở top-level:
#     PIL, cv2, numpy, torch, typing_extensions — Colab đã có hết.
#     thop/polars/nvidia-ml-py/cloudpickle/matplotlib chỉ dùng ở các nhánh
#     train/val/export, nhưng vẫn cài để code path nào import muộn cũng không vỡ.
log('pip: ultralytics (--no-deps) + dependency còn thiếu')
pip('--no-deps', 'ultralytics>=8.3.162')
pip('typing_extensions', 'ultralytics-thop', 'polars', 'nvidia-ml-py', 'cloudpickle')

# ---------- symlink thư mục model ----------
pairs = [
    (f'{COMFY}/models/unet',            P['gguf']),
    (f'{COMFY}/models/diffusion_models', P['gguf']),
    (f'{COMFY}/models/clip',            P['clip']),
    (f'{COMFY}/models/text_encoders',   P['clip']),
    (f'{COMFY}/models/vae',             P['vae']),
    (f'{COMFY}/models/ultralytics',     os.path.dirname(P['yolo'])),
    (f'{COMFY}/models/sams',            P['sam']),
    (f'{COMFY}/models/upscale_models',  P['upscale']),
]
log('Symlink models/')
for path, dest in pairs:
    os.makedirs(dest, exist_ok=True)
    if os.path.islink(path) or os.path.exists(path):
        subprocess.run(['rm', '-rf', path])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    os.symlink(dest, path)
    log(f'   {path} → {dest}')

# ---------- KIỂM TRA (bản cũ không có bước này nên lỗi âm thầm) ----------
def pkg_version(dist, module=None):
    """Lấy version package, KHÔNG BAO GIỜ crash.

    Nhiều package không có thuộc tính __version__ (vd `gguf`) → phải hỏi
    importlib.metadata (đọc dist-info do pip ghi), rồi mới fallback sang attribute.
    """
    try:
        from importlib.metadata import version, PackageNotFoundError
        try:
            return version(dist)
        except PackageNotFoundError:
            pass
    except Exception:
        pass
    try:
        v = getattr(__import__(module or dist), '__version__', None)
        return str(v) if v else None
    except Exception:
        return None


def vtuple(v):
    """'0.19.0rc1' → (0, 19, 0). Chỉ lấy phần số đầu mỗi đoạn, không crash."""
    out = []
    for seg in str(v or '').split('.')[:3]:
        m = re.match(r'\d+', seg.strip())
        out.append(int(m.group()) if m else 0)
    return tuple(out) or (0,)


# (tên pip, tên module để import, lý do cần)
CAN = [('gguf', 'gguf', 'ComfyUI-GGUF: UnetLoaderGGUF/DualCLIPLoaderGGUF'),
       ('sentencepiece', 'sentencepiece', 'tokenizer T5 của GGUF'),
       ('scikit-image', 'skimage', 'Impact Pack (FaceDetailer) — thiếu là pack raise khi import'),
       ('piexif', 'piexif', 'Impact Pack'),
       ('dill', 'dill', 'Impact Pack'),
       ('segment-anything', 'segment_anything', 'SAMLoader'),
       ('matplotlib', 'matplotlib', 'Impact Subpack (UltralyticsDetectorProvider)'),
       ('ultralytics', 'ultralytics', 'Impact Subpack (YOLO)')]

log('Kiểm tra import các package node cần:')
missing = []
for dist, mod, why in CAN:
    try:
        __import__(mod)
        log(f'   ✅ {dist} {pkg_version(dist, mod) or "không rõ version"}  ({why})')
    except Exception as e:
        missing.append(f'{dist} ({why}): {e}')
        log(f'   ❌ {dist}: {e}')

if missing:
    raise RuntimeError('❌ Thiếu dependency, ComfyUI sẽ thiếu node:\n  - ' + '\n  - '.join(missing))

# Phiên bản tối thiểu — chỉ CẢNH BÁO, không dừng cell
for dist, need in [('gguf', (0, 13))]:
    v = pkg_version(dist)
    if v is None:
        log(f'   ⚠️ Không đọc được version của {dist} — bỏ qua kiểm tra tối thiểu')
    elif vtuple(v) < need:
        n = '.'.join(str(x) for x in need)
        log(f'   ⚠️ {dist} {v} < {n} — nâng cấp: pip install -U "{dist}>={n}"')
log('✅ Xong Cell 2 → chạy Cell 3')


In [ ]:
# @title ⬇️ CELL 3 — Tải model (~12 GB, có mirror dự phòng)
BO_QUA_MODEL = False  # @param {type:"boolean"}
TAI_TAESD = True  # @param {type:"boolean"}
SO_LUONG_TAI_SONG_SONG = 3  # @param {type:"integer"}

import os, sys, json, time, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

P = json.load(open('/content/mode_ai_paths.json'))

def ok(path, minb):
    return os.path.isfile(path) and os.path.getsize(path) >= minb

def curl(url, path, minb):
    tmp = path + '.part'
    cmd = ['curl', '-L', '--fail', '--retry', '5', '--retry-delay', '2', '--retry-all-errors',
           '-C', '-', '--connect-timeout', '30', '-A', 'Mozilla/5.0', '-o', tmp, url]
    r = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if r.returncode != 0 or not os.path.isfile(tmp):
        return False
    if os.path.getsize(tmp) < minb:
        os.remove(tmp)
        return False
    shutil.move(tmp, path)
    return True

def hf(repo, filename, path, minb):
    """Tải qua huggingface_hub. KHÔNG dùng resume_download (đã bị bỏ từ hub 1.x → TypeError)."""
    from huggingface_hub import hf_hub_download
    local = hf_hub_download(repo_id=repo, filename=filename,
                            local_dir=os.path.dirname(path) or '.')
    if not os.path.isfile(local) or os.path.getsize(local) < minb:
        return False
    dest = os.path.join(os.path.dirname(path), os.path.basename(local))
    if os.path.abspath(dest) != os.path.abspath(local):
        shutil.move(local, dest)
    return True

def get(label, path, minb, sources, required=True):
    if ok(path, minb):
        log(f'✅ {label} đã có ({os.path.getsize(path)/1e6:.0f} MB)')
        return True
    log(f'⬇️  {label}')
    last = None
    for kind, *rest in sources:
        try:
            good = curl(rest[0], path, minb) if kind == 'curl' else hf(rest[0], rest[1], path, minb)
            if good:
                log(f'✅ {label} ({os.path.getsize(path)/1e6:.0f} MB)')
                return True
        except Exception as e:
            last = str(e).splitlines()[0][:160]
            log(f'   mirror lỗi: {last}')
    msg = f'❌ Không tải được {label}' + (f' — {last}' if last else '')
    if required:
        raise RuntimeError(msg + '\n   Thử tick USE_HF_MIRROR ở Cell 1 rồi chạy lại.')
    log(msg + ' — bỏ qua')
    return False

def hf_url(repo, fn):
    return f'https://huggingface.co/{repo}/resolve/main/{fn}?download=true'

def mirror_url(repo, fn):
    return f'https://hf-mirror.com/{repo}/resolve/main/{fn}?download=true'

TASKS = [
    ('FLUX UNET Q5_K_S (8.3 GB)', f"{P['gguf']}/flux1-schnell-Q5_K_S.gguf", 7_500_000_000,
     [('curl', hf_url('city96/FLUX.1-schnell-gguf', 'flux1-schnell-Q5_K_S.gguf')),
      ('curl', mirror_url('city96/FLUX.1-schnell-gguf', 'flux1-schnell-Q5_K_S.gguf')),
      ('hf', 'city96/FLUX.1-schnell-gguf', 'flux1-schnell-Q5_K_S.gguf')], True),
    ('T5-XXL Q4_K_M (2.9 GB)', f"{P['clip']}/t5-v1_1-xxl-encoder-Q4_K_M.gguf", 2_500_000_000,
     [('curl', hf_url('city96/t5-v1_1-xxl-encoder-gguf', 't5-v1_1-xxl-encoder-Q4_K_M.gguf')),
      ('curl', mirror_url('city96/t5-v1_1-xxl-encoder-gguf', 't5-v1_1-xxl-encoder-Q4_K_M.gguf')),
      ('hf', 'city96/t5-v1_1-xxl-encoder-gguf', 't5-v1_1-xxl-encoder-Q4_K_M.gguf')], True),
    ('CLIP-L (246 MB)', f"{P['clip']}/clip_l.safetensors", 200_000_000,
     [('curl', hf_url('comfyanonymous/flux_text_encoders', 'clip_l.safetensors')),
      ('curl', mirror_url('comfyanonymous/flux_text_encoders', 'clip_l.safetensors')),
      ('hf', 'comfyanonymous/flux_text_encoders', 'clip_l.safetensors')], True),
    ('FLUX VAE ae (335 MB)', f"{P['vae']}/ae.safetensors", 250_000_000,
     [('curl', hf_url('Comfy-Org/Lumina_Image_2.0_Repackaged', 'split_files/vae/ae.safetensors')),
      ('curl', mirror_url('Comfy-Org/Lumina_Image_2.0_Repackaged', 'split_files/vae/ae.safetensors')),
      ('curl', hf_url('camenduru/FLUX.1-dev', 'ae.safetensors')),
      ('hf', 'camenduru/FLUX.1-dev', 'ae.safetensors')], True),
    ('YOLO mặt face_yolov8m (52 MB)', f"{P['yolo']}/face_yolov8m.pt", 40_000_000,
     [('curl', hf_url('Bingsu/adetailer', 'face_yolov8m.pt')),
      ('curl', mirror_url('Bingsu/adetailer', 'face_yolov8m.pt')),
      ('hf', 'Bingsu/adetailer', 'face_yolov8m.pt')], True),
    ('YOLO tay hand_yolov8s (22 MB)', f"{P['yolo']}/hand_yolov8s.pt", 15_000_000,
     [('curl', hf_url('Bingsu/adetailer', 'hand_yolov8s.pt')),
      ('curl', mirror_url('Bingsu/adetailer', 'hand_yolov8s.pt')),
      ('hf', 'Bingsu/adetailer', 'hand_yolov8s.pt')], True),
    ('SAM ViT-B (375 MB)', f"{P['sam']}/sam_vit_b_01ec64.pth", 300_000_000,
     [('curl', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'),
      ('curl', hf_url('segments-ai/sam_vit_b', 'sam_vit_b_01ec64.pth')),
      ('hf', 'segments-ai/sam_vit_b', 'sam_vit_b_01ec64.pth')], True),
]
if TAI_TAESD:
    # ComfyUI tìm file BẮT ĐẦU bằng tên trong latent_format.taesd_decoder_name
    # (FLUX.1 = "taef1_decoder", xem comfy/latent_formats.py). Repo HF `madebyollin/taesd`
    # KHÔNG có file taef1 → lấy từ GitHub hoặc repo mirror.
    TASKS.append(('TAESD preview FLUX (2.5 MB, xem trước nét khi đang sample)',
                  '/content/ComfyUI/models/vae_approx/taef1_decoder.pth', 1_500_000,
                  [('curl', 'https://github.com/madebyollin/taesd/raw/main/taef1_decoder.pth'),
                   ('curl', hf_url('UmeAiRT/ComfyUI-Auto-Installer-Assets',
                                   'models/vae_approx/taef1_decoder.safetensors')),
                   ('hf', 'UmeAiRT/ComfyUI-Auto-Installer-Assets',
                    'models/vae_approx/taef1_decoder.safetensors')], False))

os.makedirs('/content/ComfyUI/models/vae_approx', exist_ok=True)
t0 = time.time()
if not BO_QUA_MODEL:
    with ThreadPoolExecutor(max_workers=max(1, int(SO_LUONG_TAI_SONG_SONG))) as ex:
        results = list(ex.map(lambda t: get(*t), TASKS))
    if not all(results):
        raise RuntimeError('❌ Có model bắt buộc chưa tải được — xem log phía trên')
else:
    log('BO_QUA_MODEL=True — chỉ kiểm tra file đã có')
    for label, path, minb, _s, req in TASKS:
        log(('✅ ' if ok(path, minb) else ('❌ ' if req else '⚠️ ')) + label
            + (f' ({os.path.getsize(path)/1e6:.0f} MB)' if os.path.isfile(path) else ' — KHÔNG CÓ'))

total = 0
for label, path, minb, _s, _r in TASKS:
    if os.path.isfile(path):
        total += os.path.getsize(path)
log(f'🟢 Tổng model: {total/1e9:.2f} GB — tải trong {time.time()-t0:.0f}s')
log('✅ Xong Cell 3 → chạy Cell 4')


In [ ]:
# @title 🧠 CELL 4 — Ghi workflow tối ưu vào máy (nhúng sẵn, không phụ thuộc repo ngoài)
WORKFLOW_DIR = "/content/workflows"  # @param {type:"string"}
TAI_BAN_UI_TU_REPO = True  # @param {type:"boolean"}

import os, json, subprocess

# ==== BEGIN BUILD_WORKFLOWS (nhúng từ scripts/build_workflows.py) ====
"""Khối sinh workflow — NHÚNG TỪ scripts/build_workflows.py (đừng sửa tay ở đây, hãy sửa file gốc rồi chạy lại scripts/make_notebook.py)."""
from typing import Any, Dict

# ----------------------------------------------------------------- model (khớp Cell 3)
UNET = "flux1-schnell-Q5_K_S.gguf"
CLIP_L = "clip_l.safetensors"
T5 = "t5-v1_1-xxl-encoder-Q4_K_M.gguf"
VAE = "ae.safetensors"
YOLO_FACE = "bbox/face_yolov8m.pt"          # Impact Subpack BẮT BUỘC tiền tố bbox/
YOLO_HAND = "bbox/hand_yolov8s.pt"
SAM = "sam_vit_b_01ec64.pth"

# ----------------------------------------------------------------- tham số tối ưu T4 16GB
BASE_STEPS = 4          # FLUX.1-schnell là model distill 4 bước
BASE_CFG = 1.0          # schnell không dùng CFG → bỏ luôn nhánh negative khi sample
DETAIL_STEPS = 4
FACE_DENOISE = 0.22
HAND_DENOISE = 0.28
HIRES_DENOISE = 0.35

# KHÔNG viết "five fingers" / "perfect hands": càng nhấn số ngón, model chưng cất càng
# hay sinh thêm ngón. Tả tay đang ở đâu/đang cầm gì (hoặc cho tay ra khỏi khung).
POS_DEFAULT = (
    "photorealistic portrait of a young Vietnamese woman, natural skin texture with visible pores, "
    "soft window light, 85mm lens, shallow depth of field, detailed eyes, hands resting out of "
    "frame, casual linen shirt, warm neutral background, film grain, high detail"
)
# schnell + cfg 1.0 → samplers.py:610 bỏ hẳn nhánh negative, nên mặc định để trống
# (đỡ một lần encode T5). Cell 6 tự điền + tự nâng cfg khi người dùng chọn dùng negative.
NEG_DEFAULT = ""


# ----------------------------------------------------------------- khối node dùng chung
def loaders() -> Dict[str, Any]:
    """1 lần nạp model, dùng lại cho mọi node phía sau (tiết kiệm VRAM + thời gian)."""
    return {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET}},
        "2": {"class_type": "DualCLIPLoaderGGUF", "inputs": {
            "clip_name1": CLIP_L, "clip_name2": T5, "type": "flux"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": VAE}},
    }


def cond(pos: str = POS_DEFAULT, neg: str = NEG_DEFAULT) -> Dict[str, Any]:
    return {
        "5": {"class_type": "CLIPTextEncode", "inputs": {"text": pos, "clip": ["2", 0]}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": ["2", 0]}},
    }


def latent(nid: str = "4", width: int = 1024, height: int = 1024,
           batch: int = 1) -> Dict[str, Any]:
    return {nid: {"class_type": "EmptyLatentImage",
                  "inputs": {"width": width, "height": height, "batch_size": batch}}}


def ksampler(nid: str, latent_id: str = "4", seed: int = 0, steps: int = BASE_STEPS,
             cfg: float = BASE_CFG, denoise: float = 1.0, model: str = "1",
             pos: str = "5", neg: str = "6") -> Dict[str, Any]:
    return {nid: {"class_type": "KSampler", "inputs": {
        "seed": seed, "steps": steps, "cfg": cfg,
        "sampler_name": "euler", "scheduler": "simple", "denoise": denoise,
        "model": [model, 0], "positive": [pos, 0], "negative": [neg, 0],
        "latent_image": [latent_id, 0]}}}


def detailer(nid: str, image_id: str, detector_id: str, seed: int, denoise: float,
             feather: int, crop_factor: float, threshold: float = 0.5, dilation: int = 8,
             sam_id: str | None = None, guide_size: int = 384, max_size: int = 768,
             steps: int = DETAIL_STEPS) -> Dict[str, Any]:
    """FaceDetailer đầy đủ input required (thiếu `wildcard`/`cycle` là ComfyUI báo lỗi)."""
    ins: Dict[str, Any] = {
        "image": [image_id, 0], "model": ["1", 0], "clip": ["2", 0], "vae": ["3", 0],
        "guide_size": guide_size, "guide_size_for": True, "max_size": max_size,
        "seed": seed, "steps": steps, "cfg": BASE_CFG,
        "sampler_name": "euler", "scheduler": "simple",
        "positive": ["5", 0], "negative": ["6", 0],
        "denoise": denoise, "feather": feather, "noise_mask": True, "force_inpaint": True,
        "bbox_threshold": threshold, "bbox_dilation": dilation,
        "bbox_crop_factor": crop_factor,
        "sam_detection_hint": "center-1", "sam_dilation": 0, "sam_threshold": 0.93,
        "sam_bbox_expansion": 0, "sam_mask_hint_threshold": 0.7,
        "sam_mask_hint_use_negative": "False",
        "drop_size": 10, "bbox_detector": [detector_id, 0], "wildcard": "", "cycle": 1,
    }
    if sam_id:
        ins["sam_model_opt"] = [sam_id, 0]
    # tiled encode/decode: crop nhỏ nên không tốn VRAM, tránh OOM trên T4
    ins["tiled_encode"] = True
    ins["tiled_decode"] = True
    return {nid: {"class_type": "FaceDetailer", "inputs": ins}}


def detector(nid: str, model_name: str) -> Dict[str, Any]:
    return {nid: {"class_type": "UltralyticsDetectorProvider",
                  "inputs": {"model_name": model_name}}}


def sam_loader(nid: str = "13") -> Dict[str, Any]:
    return {nid: {"class_type": "SAMLoader",
                  "inputs": {"model_name": SAM, "device_mode": "AUTO"}}}


def save(nid: str, prefix: str, from_id: str) -> Dict[str, Any]:
    return {nid: {"class_type": "SaveImage",
                  "inputs": {"filename_prefix": prefix, "images": [from_id, 0]}}}


# ----------------------------------------------------------------- 5 pipeline
def build_fast(width: int = 1024, height: int = 1024,
               pos: str = POS_DEFAULT, neg: str = NEG_DEFAULT) -> Dict[str, Any]:
    """Nhanh nhất: 4 bước, không detailer. ~15-20s/ảnh 1024² trên T4."""
    wf: Dict[str, Any] = {}
    wf.update(loaders())
    wf.update(cond(pos, neg))
    wf.update(latent("4", width, height))
    wf.update(ksampler("7"))
    wf["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
    wf.update(save("9", "flux/fast", "8"))
    return wf


def build_standard(width: int = 1024, height: int = 1024,
                   pos: str = POS_DEFAULT, neg: str = NEG_DEFAULT) -> Dict[str, Any]:
    """Mặc định: 4 bước + sửa mặt bằng SAM. ~25-35s/ảnh."""
    wf = build_fast(width, height, pos, neg)
    wf.update(detector("10", YOLO_FACE))
    wf.update(sam_loader("13"))
    wf.update(save("30", "flux/base", "8"))
    wf.update(detailer("20", "8", "10", seed=100, denoise=FACE_DENOISE,
                       feather=8, crop_factor=3.0, sam_id="13"))
    wf["9"] = save("9", "flux/standard", "20")["9"]
    return wf


def build_quality(width: int = 1024, height: int = 1024,
                  pos: str = POS_DEFAULT, neg: str = NEG_DEFAULT) -> Dict[str, Any]:
    """Chất lượng: 4 bước + sửa mặt (SAM) + sửa tay. ~40-55s/ảnh."""
    wf = build_standard(width, height, pos, neg)
    wf.update(detector("11", YOLO_HAND))
    wf.update(detailer("21", "20", "11", seed=101, denoise=HAND_DENOISE,
                       feather=16, crop_factor=2.5, threshold=0.35, dilation=6))
    wf["9"] = save("9", "flux/quality", "21")["9"]
    return wf


def build_hires(width: int = 832, height: int = 1216,
                pos: str = POS_DEFAULT, neg: str = NEG_DEFAULT) -> Dict[str, Any]:
    """Ảnh nét cỡ lớn: base + sửa mặt, upscale 1.5× rồi lấy lại chi tiết bằng 4 bước
    denoise thấp. Chỉ dùng node có sẵn trong ComfyUI (không cần ESRGAN/UltimateSDUpscale)."""
    wf = build_standard(width, height, pos, neg)
    wf["40"] = {"class_type": "ImageScaleBy", "inputs": {
        "image": ["20", 0], "upscale_method": "lanczos", "scale_by": 1.5}}
    wf["41"] = {"class_type": "VAEEncode", "inputs": {"pixels": ["40", 0], "vae": ["3", 0]}}
    wf.update(ksampler("42", latent_id="41", seed=200, denoise=HIRES_DENOISE))
    wf["43"] = {"class_type": "VAEDecode", "inputs": {"samples": ["42", 0], "vae": ["3", 0]}}
    wf["9"] = save("9", "flux/hires", "43")["9"]
    wf["30"] = save("30", "flux/hires_base", "20")["30"]
    return wf


def build_inpaint(image: str = "input_image.png", mask: str = "mask.png",
                  pos: str = ("a natural human hand resting flat on a surface, realistic skin "
                              "texture, even fingernails, photorealistic, sharp focus"),
                  neg: str = NEG_DEFAULT, denoise: float = 0.5,
                  steps: int = 6, grow_mask_by: int = 12) -> Dict[str, Any]:
    """Sửa vùng tô (tay/mặt/chân) trên ảnh có sẵn."""
    wf: Dict[str, Any] = {}
    wf.update(loaders())
    wf.update(cond(pos, neg))
    wf["4"] = {"class_type": "LoadImage", "inputs": {"image": image}}
    wf["5m"] = {"class_type": "LoadImage", "inputs": {"image": mask}}
    wf["6m"] = {"class_type": "ImageToMask", "inputs": {"image": ["5m", 0], "channel": "red"}}
    wf["9e"] = {"class_type": "VAEEncodeForInpaint", "inputs": {
        "pixels": ["4", 0], "vae": ["3", 0], "mask": ["6m", 0],
        "grow_mask_by": grow_mask_by}}
    wf.update(ksampler("7", latent_id="9e", steps=steps, denoise=denoise))
    wf["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
    wf.update(save("10", "flux/inpaint", "8"))
    return wf


PIPELINES = {
    "flux_q5_fast": build_fast,
    "flux_q5_standard": build_standard,
    "flux_q5_quality": build_quality,
    "flux_q5_hires": build_hires,
    "flux_q5_inpaint": build_inpaint,
}


def build_all() -> Dict[str, Dict[str, Any]]:
    return {name: fn() for name, fn in PIPELINES.items()}
# ==== END BUILD_WORKFLOWS ====

# ==== BEGIN PROMPT_PRESETS (nhúng từ scripts/prompt_presets.py) ====
"""Khối thư viện prompt — NHÚNG TỪ scripts/prompt_presets.py (đừng sửa tay ở đây)."""

NEGATIVE = ""

# --- ngưỡng CFG (căn cứ: comfy/samplers.py:610 bỏ nhánh negative ở cfg=1.0) -------
CFG_MAC_DINH = 1.0      # nhanh nhất, nhưng negative KHÔNG được đọc
CFG_NEG_HIEU_LUC = 2.0  # mức thấp nhất để negative bắt đầu có tác dụng
STEPS_TOI_THIEU_CFG = 8  # cfg>1 trên model chưng cất 4 bước dễ "cháy" → cần thêm bước
GIOI_HAN_TU = 60        # quá số này prompt bắt đầu loãng ý

# --- Thư viện negative: mỗi khối nhắm vào một NHÓM lỗi ----------------------------
# Chỉ có tác dụng khi cfg > 1.0. Giữ ngắn: T5 encode cả cụm này mỗi lần chạy.
NEG_LIBRARY = {
    "chung": (
        "blurry, low resolution, jpeg artifacts, bad anatomy, deformed, disfigured, "
        "extra limbs, mutated hands, distorted face, asymmetric eyes, cross-eyed, "
        "watermark, signature, logo, text, oversaturated, plastic skin, duplicate subject"
    ),
    "tay": (
        "extra fingers, fused fingers, missing fingers, too many fingers, mutated hands, "
        "deformed hands, twisted wrists, broken fingernails, hands merging into objects, "
        "blurry hands, hands growing out of sleeves"
    ),
    "mat": (
        "asymmetric eyes, cross-eyed, extra eyes, deformed pupils, melted face, warped mouth, "
        "distorted nose, over-smoothed skin, uncanny plastic face, double face, blurry face"
    ),
    "chu": (
        "text, letters, watermark, signature, logo, caption, subtitle, UI overlay, "
        "garbled characters, random symbols, misspelled words"
    ),
    "co_the": (
        "extra arms, extra legs, extra heads, mutated limbs, disconnected limbs, floating "
        "limbs, twisted torso, unnatural proportions, duplicate body parts, fused bodies"
    ),
}

# --- Các chế độ negative hiện trong ô NEG_MODE của Cell 6 --------------------------
# (id, nhãn tiếng Việt). id khớp key của NEG_LIBRARY, trừ các id đặc biệt:
#   theo    = dùng negative gắn sẵn trong preset
#   tat_ca  = ghép mọi khối (dài nhất)
#   tu_viet = lấy chuỗi người dùng gõ ở ô NEGATIVE_PROMPT
#   khong   = tắt negative (nhanh nhất, về đúng cfg=1.0)
NEG_MODES = [
    ("theo", "theo preset"),
    ("chung", "chung - chống lỗi tổng quát"),
    ("tay", "tay - lỗi bàn tay"),
    ("mat", "mat - lỗi khuôn mặt"),
    ("chu", "chu - chữ/ký tự rác"),
    ("co_the", "co_the - thừa chi/cơ thể"),
    ("tat_ca", "tat_ca - tất cả"),
    ("tu_viet", "tu_viet - tự viết ở dưới"),
    ("khong", "khong - không dùng negative"),
]

# --- Nhãn dùng chung cho giao diện Cell 6 (để dropdown và code không bao giờ lệch) --
TUY_CHON = "(tự viết prompt ở dưới)"
SIZE_THEO_PRESET = "theo preset (khuyên dùng)"

# --- Khung hình khuyên dùng: giữ ~1 megapixel ---------------------------------------
SIZES = {
    "832x1216 (dọc, ~1MP)": [832, 1216],
    "1216x832 (ngang, ~1MP)": [1216, 832],
    "1024x1024 (vuông, ~1MP)": [1024, 1024],
    "768x1024 (dọc nhỏ, nhanh)": [768, 1024],
    "1024x768 (ngang nhỏ, nhanh)": [1024, 768],
    "1344x768 (ngang rộng, ~1MP)": [1344, 768],
}

# --- Cụm rủi ro: Cell 6 quét prompt DƯƠNG và cảnh báo trước khi chạy ---------------
CANH_BAO = [
    ("five fingers", "Đếm ngón khiến model chưng cất hay sinh THÊM ngón. Tả tay đang cầm/giấu gì."),
    ("ten fingers", "Đếm ngón khiến model chưng cất hay sinh THÊM ngón. Tả tay đang cầm/giấu gì."),
    ("perfect hands", "Nhấn 'perfect' không làm tay đẹp hơn. Hãy tả tư thế tay cụ thể."),
    ("detailed fingers", "Càng nhấn chi tiết ngón, ngón càng hay lỗi. Tả vật tay đang cầm."),
    ("perfect anatomy", "Cụm này vô nghĩa với FLUX; thay bằng mô tả tư thế/đạo cụ."),
    ("masterpiece", "Tag kiểu SDXL, không ăn với FLUX — chỉ làm prompt dài thêm."),
    ("best quality", "Tag kiểu SDXL, không ăn với FLUX — chỉ làm prompt dài thêm."),
    ("8k", "Tag độ phân giải kiểu SDXL, không ăn với FLUX."),
    ("ultra detailed", "Tag kiểu SDXL, không ăn với FLUX — mô tả chi tiết cụ thể thì tốt hơn."),
]

# Mức rủi ro sinh lỗi giải phẫu, để người dùng chọn biết mà liệu
THAP = "Thấp"
TRUNG_BINH = "Trung bình"
CAO = "Cao"

PRESETS = [
    {
        "id": "chan_dung_can",
        "ten": "Chân dung cận cảnh — ít lỗi nhất",
        "prompt": (
            "Close-up portrait of a young Vietnamese woman, natural skin with visible pores, "
            "soft window light from the left, 85mm lens, shallow depth of field, "
            "head and shoulders framing, plain warm backdrop, subtle film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_standard",
        "rui_ro": THAP,
        "negative_keys": ["chung", "mat"],
        "ghi_chu": "Không có tay trong khung → gần như không có lỗi giải phẫu.",
    },
    {
        "id": "ban_than_cam_coc",
        "ten": "Bán thân, hai tay cầm cốc",
        "prompt": (
            "Half-body portrait of a young Vietnamese woman sitting in a cafe, "
            "both hands wrapped around a ceramic coffee cup resting on the table, "
            "natural skin texture, soft afternoon window light, 50mm lens, "
            "shallow depth of field, blurred cafe background, film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_quality",
        "rui_ro": TRUNG_BINH,
        "negative_keys": ["chung", "tay"],
        "ghi_chu": "Tay có vật bám (cốc) → render ổn định hơn tay trôi nổi.",
    },
    {
        "id": "toan_than_tui_quan",
        "ten": "Toàn thân đứng, tay trong túi",
        "prompt": (
            "Full-body photograph of a young Vietnamese woman standing on a quiet street, "
            "hands tucked into the pockets of a beige trench coat, relaxed natural pose, "
            "soft golden hour light, 35mm lens, full figure in frame from head to shoes, "
            "shallow depth of field, film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_standard",
        "rui_ro": THAP,
        "negative_keys": ["chung", "co_the"],
        "ghi_chu": "Tay giấu trong túi → không lộ ngón tay.",
    },
    {
        "id": "toan_than_ngoi",
        "ten": "Toàn thân ngồi, tay đan trên đùi",
        "prompt": (
            "Full-body photograph of a young Vietnamese woman sitting on a wooden bench "
            "in a park, hands clasped together resting on her lap, relaxed posture, "
            "dappled sunlight through leaves, 35mm lens, entire figure in frame, "
            "natural colors, film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_standard",
        "rui_ro": THAP,
        "negative_keys": ["chung", "co_the"],
        "ghi_chu": "Tay đan thành một khối kín → dễ render, ít lỗi ngón.",
    },
    {
        "id": "thoi_trang",
        "ten": "Thời trang, tay xách túi",
        "prompt": (
            "Fashion editorial photograph of a Vietnamese woman in a flowing white dress "
            "standing against a textured concrete wall, one hand holding a small leather "
            "handbag at her side, dramatic side lighting, 85mm lens, full body in frame, "
            "high fashion magazine aesthetic, film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_quality",
        "rui_ro": TRUNG_BINH,
        "negative_keys": ["chung", "tay", "co_the"],
        "ghi_chu": "Tay xách túi có điểm tựa; vẫn nên chạy quality để sửa tay.",
    },
    {
        "id": "duong_pho",
        "ten": "Đời thường đường phố, xách túi tote",
        "prompt": (
            "Candid street photograph of a young Vietnamese woman walking through a "
            "Hanoi old quarter street, carrying a canvas tote bag in her right hand, "
            "natural walking pose, overcast soft light, 35mm lens, full body in frame, "
            "documentary photography style, film grain"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_quality",
        "rui_ro": TRUNG_BINH,
        "negative_keys": ["chung", "tay", "co_the"],
        "ghi_chu": "Đi bộ + túi xách: tay có việc để làm, ít sinh ngón thừa.",
    },
    {
        "id": "phong_canh",
        "ten": "Phong cảnh (không có người)",
        "prompt": (
            "Wide landscape photograph of terraced rice fields in northern Vietnam at "
            "sunrise, mist drifting between the hills, warm golden light, 24mm wide angle, "
            "deep depth of field, high detail, no people"
        ),
        "size": [1216, 832],
        "pipeline": "flux_q5_fast",
        "rui_ro": THAP,
        "negative_keys": ["chung", "chu"],
        "ghi_chu": "Không có người → không có lỗi giải phẫu. 'no people' là cụm rất hiệu lực.",
    },
    {
        "id": "san_pham",
        "ten": "Sản phẩm studio",
        "prompt": (
            "Studio product photograph of a matte black ceramic coffee mug on a light oak "
            "table, soft diffused lighting from the left, subtle shadow, seamless light grey "
            "backdrop, 100mm macro lens, sharp focus, commercial photography"
        ),
        "size": [1024, 1024],
        "pipeline": "flux_q5_fast",
        "rui_ro": THAP,
        "negative_keys": ["chung", "chu"],
        "ghi_chu": "Vật vô tri → không có lỗi giải phẫu, dùng fast cho nhanh.",
    },
    {
        "id": "anh_minh_hoa",
        "ten": "Minh họa anime",
        "prompt": (
            "Anime illustration of a girl with long silver hair and aqua eyes in a school "
            "uniform, standing under cherry blossoms, soft afternoon sunlight, clean linework, "
            "cel shading, detailed background, studio anime key visual"
        ),
        "size": [832, 1216],
        "pipeline": "flux_q5_fast",
        "rui_ro": CAO,
        "negative_keys": ["chung"],
        "ghi_chu": ("YOLO mặt (face_yolov8m.pt) được huấn luyện trên mặt người thật nên "
                    "có thể KHÔNG nhận diện được mặt anime → FaceDetailer vô tác dụng. "
                    "Dùng fast, hoặc tự sửa bằng Cell 7."),
    },
]

# Những cụm nên TRÁNH — kèm lý do
ANTI_PATTERNS = [
    ("five fingers, perfect hands, detailed fingers",
     "Nhấn mạnh số ngón khiến model chưng cất hay sinh THÊM ngón. Hãy tả tay đang làm gì."),
    ("best quality, masterpiece, 8k, ultra detailed",
     "Tag chất lượng kiểu SDXL không ăn với FLUX; làm prompt dài mà không thêm chi tiết gì."),
    ("Negative prompt mà vẫn để cfg = 1.0",
     "comfy/samplers.py:610 bỏ hẳn nhánh negative ở cfg=1.0. Muốn negative có tác dụng: "
     "nâng ô CFG trong Cell 6 lên 2.0 trở lên (kèm STEPS >= 8)."),
    ("Prompt quá 60 từ",
     "T5-XXL cắt ở 512 token, nhưng prompt dài làm loãng ý chính. Giữ 30-45 từ."),
    ("Kích thước quá xa 1 megapixel",
     "VD 512×512 hay 2048×2048: schnell sinh lỗi cấu trúc. Giữ ~1MP: 832×1216 / 1024²."),
]


def _gop(cac_khoi: list[str]) -> str:
    """Ghép nhiều khối negative thành một chuỗi, bỏ trùng, giữ thứ tự."""
    seen, out = set(), []
    for key in cac_khoi:
        for cum in NEG_LIBRARY.get(key, "").split(","):
            cum = cum.strip()
            if cum and cum.lower() not in seen:
                seen.add(cum.lower())
                out.append(cum)
    return ", ".join(out)


def build() -> dict:
    # negative của từng preset = ghép các khối trong NEG_LIBRARY (một nguồn, không gõ lại)
    presets = []
    for p in PRESETS:
        p = dict(p)
        keys = p.pop("negative_keys", [])
        p["negative"] = _gop(keys) if keys else NEGATIVE
        p["negative_keys"] = keys
        presets.append(p)
    return {
        "negative": NEGATIVE,
        "cfg_mac_dinh": CFG_MAC_DINH,
        "cfg_neg_hieu_luc": CFG_NEG_HIEU_LUC,
        "steps_toi_thieu_cfg": STEPS_TOI_THIEU_CFG,
        "gioi_han_tu": GIOI_HAN_TU,
        "neg_library": NEG_LIBRARY,
        "neg_modes": [{"id": i, "ten": t} for i, t in NEG_MODES],
        "sizes": SIZES,
        "canh_bao": [{"cum": a, "ly_do": b} for a, b in CANH_BAO],
        "luu_y": [
            "cfg=1.0 trên FLUX.1-schnell → negative prompt KHÔNG được đọc (comfy/samplers.py:610).",
            "Muốn negative có tác dụng: nâng CFG >= 2.0 trong Cell 6 (kèm STEPS >= 8, chậm hơn).",
            "Negative chỉ vá được một phần — cách tránh lỗi tay vẫn là tả tay đang cầm/giấu/đan.",
            "Viết câu tự nhiên: chủ thể → tư thế → bối cảnh → ánh sáng → ống kính → khung hình.",
            "Giữ ~1 megapixel: 832×1216 (dọc), 1216×832 (ngang), 1024×1024 (vuông).",
        ],
        "anti_patterns": [{"cum": a, "ly_do": b} for a, b in ANTI_PATTERNS],
        "presets": presets,
    }
# ==== END PROMPT_PRESETS ====

os.makedirs(WORKFLOW_DIR, exist_ok=True)
COMFY = '/content/ComfyUI'
written = []
for name, wf in build_all().items():
    for dest_dir in (WORKFLOW_DIR, f'{COMFY}/input'):
        os.makedirs(dest_dir, exist_ok=True)
        path = os.path.join(dest_dir, f'{name}.json')
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(wf, f, ensure_ascii=False, indent=1)
        written.append(path)
    print(f'✅ {name}.json ({len(wf)} node)')

# thư viện prompt — Cell 6 đọc để hiện danh sách chọn
prompts_doc = {'negative': NEGATIVE,
               'luu_y': [f'cfg=1.0 trên FLUX.1-schnell → negative KHÔNG được dùng'],
               'anti_patterns': [{'cum': a, 'ly_do': b} for a, b in ANTI_PATTERNS],
               'presets': PRESETS}
for dest_dir in (WORKFLOW_DIR, f'{COMFY}/input'):
    os.makedirs(dest_dir, exist_ok=True)
    with open(os.path.join(dest_dir, 'prompts.json'), 'w', encoding='utf-8') as f:
        json.dump(prompts_doc, f, ensure_ascii=False, indent=1)
print(f'✅ prompts.json ({len(PRESETS)} preset)')

# bản UI (có layout, kéo-thả vào giao diện) — tải từ repo, không có thì bỏ qua
if TAI_BAN_UI_TU_REPO:
    REPO = 'manhlee1196-boop/mode-ai'
    BRANCHES = ['main', 'arena/01a0d3b5-mode-ai']
    ui_dir = f'{COMFY}/user/default/workflows'
    os.makedirs(ui_dir, exist_ok=True)
    for br in BRANCHES:
        got = 0
        for name in build_all():
            url = f'https://raw.githubusercontent.com/{REPO}/{br}/workflows/ui/{name}.json'
            dest = os.path.join(ui_dir, f'{name}.json')
            r = subprocess.run(['curl', '-sfL', '--max-time', '20', '-o', dest, url])
            if r.returncode == 0 and os.path.getsize(dest) > 200:
                got += 1
            elif os.path.isfile(dest):
                os.remove(dest)
        if got:
            print(f'✅ {got} workflow bản UI (layout) từ branch {br} → {ui_dir}')
            break
    else:
        print('ℹ️ Không tải được bản UI từ repo — dùng bản API (giao diện ComfyUI mới '
              'vẫn mở được, chỉ không có layout) hoặc dùng Cell 6.')

print('\nCách dùng:')
print('  • Cell 6: tạo ảnh ngay trong Colab, không cần mở giao diện')
print('  • Hoặc mở link ComfyUI → Workflow → Open → chọn file trong /content/workflows')
print('✅ Xong Cell 4 → chạy Cell 5')


In [ ]:
# @title 🚀 CELL 5 — Khởi chạy ComfyUI + tunnel
PORT = 8188  # @param {type:"integer"}
TUNNEL = "cloudflared http2"  # @param ["cloudflared http2", "cloudflared quic", "không tunnel"]
VRAM_MODE = "Mặc định — Dynamic VRAM (khuyến nghị)"  # @param ["Mặc định — Dynamic VRAM (khuyến nghị)", "lowvram", "normalvram", "highvram", "novram", "cpu"]
RESERVE_VRAM_GB = 1.0  # @param {type:"slider", min:0.0, max:4.0, step:0.1}
FORCE_FP16 = True  # @param {type:"boolean"}
VAE_PREC = "fp16-vae"  # @param ["fp16-vae", "fp32-vae", "cpu-vae", "Mặc định"]
ATTENTION = "pytorch (SDPA)"  # @param ["pytorch (SDPA)", "sage", "flash", "Mặc định"]
PREVIEW = "taesd"  # @param ["taesd", "auto", "latent2rgb", "none"]
CACHE_LRU = 0  # @param {type:"integer"}
FAST_FP16_ACCUM = False  # @param {type:"boolean"}
EXTRA_ARGS = ""  # @param {type:"string"}

import os, re, sys, json, time, socket, shutil, subprocess

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
    sys.stdout.flush()

COMFY = '/content/ComfyUI'
P = json.load(open('/content/mode_ai_paths.json'))
assert os.path.isfile(f'{COMFY}/main.py'), '❌ Chạy Cell 1 trước'

log('Dừng tiến trình cũ')
os.system('pkill -f "python.*main.py" >/dev/null 2>&1 || true')
os.system('pkill -f cloudflared >/dev/null 2>&1 || true')
time.sleep(2)

cmd = [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', str(int(PORT)),
       '--preview-method', PREVIEW,
       '--output-directory', f'{COMFY}/output',
       '--input-directory', f'{COMFY}/input',
       '--temp-directory', f'{COMFY}/temp',
       '--cuda-device', '0',
       '--enable-cors-header', '*',
       '--disable-xformers']

# ComfyUI >= 0.3x bật Dynamic VRAM mặc định cho NVIDIA; --lowvram lúc đó bị bỏ qua.
# Chỉ thêm cờ vram khi người dùng explicitly chọn.
vram_map = {'lowvram': '--lowvram', 'normalvram': '--normalvram', 'highvram': '--highvram',
            'novram': '--novram', 'cpu': '--cpu'}
for k, flag in vram_map.items():
    if VRAM_MODE.startswith(k):
        cmd.append(flag)
if RESERVE_VRAM_GB and RESERVE_VRAM_GB > 0 and not VRAM_MODE.startswith('cpu'):
    cmd += ['--reserve-vram', str(float(RESERVE_VRAM_GB))]
if FORCE_FP16:
    cmd.append('--force-fp16')
if VAE_PREC in ('fp16-vae', 'fp32-vae', 'cpu-vae'):
    cmd.append('--' + VAE_PREC)
att_map = {'pytorch (SDPA)': '--use-pytorch-cross-attention',
           'sage': '--use-sage-attention', 'flash': '--use-flash-attention'}
if ATTENTION in att_map:
    cmd.append(att_map[ATTENTION])
if int(CACHE_LRU) > 0:
    cmd += ['--cache-lru', str(int(CACHE_LRU))]
if FAST_FP16_ACCUM:
    cmd += ['--fast', 'fp16_accumulation']
if EXTRA_ARGS.strip():
    cmd += EXTRA_ARGS.strip().split()

log('Lệnh: ' + ' '.join(cmd))
logf = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd=COMFY)
open('/content/comfy.pid', 'w').write(str(proc.pid))

def port_open():
    try:
        with socket.create_connection(('127.0.0.1', int(PORT)), timeout=1):
            return True
    except OSError:
        return False

ok = False
for i in range(240):
    time.sleep(1)
    if proc.poll() is not None:
        os.system('tail -40 /content/comfyui.log')
        raise RuntimeError('❌ ComfyUI thoát khi khởi động — xem log ở trên')
    if port_open():
        ok = True
        break
    if i % 20 == 19:
        log(f'  ...đợi {i+1}s')
        os.system('tail -2 /content/comfyui.log')
if not ok:
    os.system('tail -40 /content/comfyui.log')
    raise RuntimeError('❌ Quá 240s chưa mở cổng')
log(f'✅ ComfyUI đang chạy cổng {PORT}')

url = None
if TUNNEL.startswith('cloudflared'):
    if not shutil.which('cloudflared'):
        log('Cài cloudflared')
        os.system('wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/'
                  'cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb')
        os.system('dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1')
    proto = 'http2' if 'http2' in TUNNEL else 'quic'
    cff = open('/content/cloudflared.log', 'w')
    cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{int(PORT)}',
                           '--http-host-header', f'127.0.0.1:{int(PORT)}', '--protocol', proto],
                          stdout=cff, stderr=subprocess.STDOUT)
    for _ in range(90):
        time.sleep(1)
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',
                      open('/content/cloudflared.log', errors='ignore').read())
        if m:
            url = m.group(0).rstrip('/')
            break
    if url:
        open('/content/comfy_url.txt', 'w').write(url)

print('\n' + '=' * 64)
if url:
    print('🎨 COMFYUI — copy link, DÁN vào tab mới (đừng bấm trong Colab):\n')
    print('   ' + url)
else:
    print(f'⚠️ Chưa có link tunnel. Local: http://127.0.0.1:{int(PORT)} (dùng Cell 5 localtunnel nếu cần)')
print('=' * 64)
print('Giờ chạy Cell 6 để tạo ảnh ngay trong Colab, hoặc Cell 8 để chẩn đoán node.')


In [ ]:
# @title 🖼 CELL 6 — Tạo ảnh ngay trong Colab (headless, không cần mở giao diện)
# ╔═════════════════════ 1) CHỌN CẢNH ═════════════════════╗
PRESET = "chan_dung_can"  # @param ["(tự viết prompt ở dưới)", "chan_dung_can", "ban_than_cam_coc", "toan_than_tui_quan", "toan_than_ngoi", "thoi_trang", "duong_pho", "phong_canh", "san_pham", "anh_minh_hoa"]
PIPELINE = "flux_q5_standard"  # @param ["flux_q5_fast", "flux_q5_standard", "flux_q5_quality", "flux_q5_hires", "flux_q5_inpaint"]

# ╔═════════════════════ 2) PROMPT DƯƠNG ═══════════════════╗
PROMPT = "Close-up portrait of a young Vietnamese woman, natural skin with visible pores, soft window light from the left, 85mm lens, shallow depth of field, head and shoulders framing, plain warm backdrop, subtle film grain"  # @param {type:"string"}
THEM_VAO_PROMPT = ""  # @param {type:"string"}

# ╔═════════════════════ 3) NEGATIVE PROMPT ═════════════════╗
# ⚠️ comfy/samplers.py:610 — ở cfg = 1.0 ComfyUI BỎ HẲN nhánh negative,
# nên muốn negative này có tác dụng phải nâng CFG (ô dưới) lên 2.0 trở lên.
NEG_MODE = "theo preset"  # @param ["theo preset", "chung - chống lỗi tổng quát", "tay - lỗi bàn tay", "mat - lỗi khuôn mặt", "chu - chữ/ký tự rác", "co_the - thừa chi/cơ thể", "tat_ca - tất cả", "tu_viet - tự viết ở dưới", "khong - không dùng negative"]
NEGATIVE_PROMPT = ""  # @param {type:"string"}
CFG = 1.0  # @param {type:"slider", min:1, max:5, step:0.5}
TU_DONG_BAT_CFG = True  # @param {type:"boolean"}

# ╔═════════════════════ 4) SAMPLING ════════════════════════╗
STEPS = 4  # @param {type:"slider", min:1, max:20, step:1}
BC_SUA_CHI_TIET = 4  # @param {type:"slider", min:2, max:12, step:1}
SAMPLER = "euler"  # @param ["euler", "euler_ancestral", "heun", "dpmpp_2m", "dpmpp_2m_sde", "lcm", "ddim", "uni_pc"]
SCHEDULER = "simple"  # @param ["simple", "normal", "beta", "karras", "sgm_uniform", "exponential"]
SEED = -1  # @param {type:"integer"}
SO_ANH = 1  # @param {type:"slider", min:1, max:4, step:1}

# ╔═════════════════════ 5) KHUNG HÌNH & ĐẦU RA ═════════════╗
SIZE = "theo preset (khuyên dùng)"  # @param ["theo preset (khuyên dùng)", "832x1216 (dọc, ~1MP)", "1216x832 (ngang, ~1MP)", "1024x1024 (vuông, ~1MP)", "768x1024 (dọc nhỏ, nhanh)", "1024x768 (ngang nhỏ, nhanh)", "1344x768 (ngang rộng, ~1MP)"]
TEN_FILE = "flux/anh"  # @param {type:"string"}
NHIEU_PROMPT = ""  # @param {type:"string"}
LUU_VAO_DRIVE = False  # @param {type:"boolean"}

import os, re, json, time, random
import requests
from PIL import Image
from IPython.display import display

COMFY = 'http://127.0.0.1:8188'
WORKFLOW_DIR = '/content/workflows'
OUT = '/content/ComfyUI/output'
IN_DIR = '/content/ComfyUI/input'
TUY_CHON = "(tự viết prompt ở dưới)"
SIZE_THEO_PRESET = "theo preset (khuyên dùng)"
SIZES = {"832x1216 (dọc, ~1MP)": [832, 1216], "1216x832 (ngang, ~1MP)": [1216, 832], "1024x1024 (vuông, ~1MP)": [1024, 1024], "768x1024 (dọc nhỏ, nhanh)": [768, 1024], "1024x768 (ngang nhỏ, nhanh)": [1024, 768], "1344x768 (ngang rộng, ~1MP)": [1344, 768]}


def _doc():
    """Đọc thư viện prompt do Cell 4 sinh ra (presets + negative + cảnh báo)."""
    try:
        with open(os.path.join(WORKFLOW_DIR, 'prompts.json'), encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print('⚠️ Không đọc được prompts.json (%s) — bỏ qua preset/negative có sẵn' % e)
        return {}


LIB = _doc()
PRESETS = {p['id']: p for p in LIB.get('presets', [])}
NEG_LIB = LIB.get('neg_library', {})
CANH_BAO = LIB.get('canh_bao', [])
CFG_NEG = float(LIB.get('cfg_neg_hieu_luc', 2.0))
STEPS_CFG = int(LIB.get('steps_toi_thieu_cfg', 8))
GIOI_HAN_TU = int(LIB.get('gioi_han_tu', 60))


def _key(nhan):
    """'tay - lỗi bàn tay' → 'tay' (đầu nhãn chính là id khối negative)."""
    return nhan.split(' - ')[0].strip().split(' ')[0]


def _noi(*cac_doan):
    """Ghép các đoạn negative, bỏ cụm trùng (giữ thứ tự, không phá dấu phẩy)."""
    out, seen = [], set()
    for doan in cac_doan:
        for cum in str(doan or '').split(','):
            cum = cum.strip().rstrip('.')
            if cum and cum.lower() not in seen:
                seen.add(cum.lower())
                out.append(cum)
    return ', '.join(out)


def gop_negative(mode, tu_viet, preset_doc):
    """Trả về chuỗi negative theo NEG_MODE (đọc danh sách từ prompts.json)."""
    key = _key(mode)
    if key == 'khong':
        return ''
    if key == 'tu_viet':
        return _noi(tu_viet)
    if key == 'theo':
        return _noi(preset_doc.get('negative', ''), tu_viet)
    if key == 'tat_ca':
        return _noi(*list(NEG_LIB.values()), tu_viet)
    return _noi(NEG_LIB.get(key, ''), tu_viet)


def apply_preset(preset_id, prompt, pipeline, them=''):
    """Chọn preset → preset thắng (prompt + pipeline + size + negative)."""
    them = (them or '').strip()
    doc = PRESETS.get(preset_id)
    if preset_id == TUY_CHON or doc is None:
        if preset_id != TUY_CHON:
            print('⚠️ Không thấy preset "%s" — giữ tuỳ chọn của bạn' % preset_id)
        pos = ('%s, %s' % (prompt.rstrip(' ,'), them)) if them else prompt
        return pos, pipeline, None, {}
    print('📋 %s' % doc['ten'])
    print('   rủi ro: %s  |  %s @ %sx%s' % (
        doc['rui_ro'], doc['pipeline'], doc['size'][0], doc['size'][1]))
    print('   %s' % doc['ghi_chu'])
    pos = ('%s, %s' % (doc['prompt'].rstrip(' ,'), them)) if them else doc['prompt']
    return pos, doc['pipeline'], (int(doc['size'][0]), int(doc['size'][1])), dict(doc)


def chon_size(nhan, preset_size, mac_dinh=(832, 1216)):
    if nhan != SIZE_THEO_PRESET and nhan in SIZES:
        return tuple(SIZES[nhan])
    return tuple(preset_size) if preset_size else mac_dinh


def xu_ly_cfg(negative, cfg, steps, tu_dong):
    """cfg = 1.0 → negative bị bỏ qua (samplers.py:610). Tự nâng nếu được phép."""
    cfg, steps, note = float(cfg), int(steps), ''
    if negative and cfg <= 1.0:
        if tu_dong:
            cfg, steps = CFG_NEG, max(steps, STEPS_CFG)
            note = ('đang dùng negative → tự nâng cfg=%.1f, steps=%d (chậm hơn cfg=1.0). '
                    'Muốn nhanh lại: NEG_MODE = "khong - không dùng negative".' % (cfg, steps))
        else:
            note = ('⚠️ cfg=1.0 → ComfyUI BỎ QUA negative (comfy/samplers.py:610). '
                    'Hãy nâng CFG hoặc bật TU_DONG_BAT_CFG.')
    elif negative and steps < STEPS_CFG:
        note = ('⚠️ cfg=%.1f mà chỉ %d bước: model chưng cất dễ ra ảnh cháy màu, '
                'nên để ít nhất %d bước.' % (cfg, steps, STEPS_CFG))
    return cfg, steps, note


def kiem_tra(prompt, w, h, cfg, steps, negative):
    """Quét prompt trước khi chạy — phát hiện cụm hay gây lỗi, đỡ mất một lượt generate."""
    low = (prompt or '').lower()
    for cb in CANH_BAO:
        if cb['cum'] in low:
            print('⚠️ Prompt có "%s": %s' % (cb['cum'], cb['ly_do']))
    n_tu = len(re.findall(r"[A-Za-z0-9'-]+", prompt or ''))
    if n_tu > GIOI_HAN_TU:
        print('⚠️ Prompt %d từ (> %d): ý chính bị loãng, nên cắt bớt.' % (n_tu, GIOI_HAN_TU))
    mp = w * h / 1e6
    if not (0.45 <= mp <= 1.7):
        print('⚠️ %dx%d = %.2f MP, xa ~1MP → schnell hay sinh lỗi cấu trúc.' % (w, h, mp))
    if negative and cfg <= 1.0:
        print('⚠️ Có negative mà cfg=1.0 → negative KHÔNG được đọc.')
    if negative and cfg > 1.0:
        print('✅ Negative đang BẬT (cfg=%.1f > 1.0, %d từ).' % (
            cfg, len(negative.split(','))))


def _health():
    try:
        return requests.get('%s/system_stats' % COMFY, timeout=5).status_code == 200
    except Exception:
        return False


def _nodes(wf, class_type):
    return [k for k, v in wf.items() if v.get('class_type') == class_type]


def _pos_neg(wf):
    """Node 5 = prompt dương, node 6 = negative (đúng quy ước builder).

    Không đoán theo độ dài text: negative dài hơn prompt sẽ làm heuristic cũ bị ngược.
    """
    enc = _nodes(wf, 'CLIPTextEncode')
    if '5' in enc:
        return '5', ('6' if '6' in enc else None)
    enc = sorted(enc, key=lambda k: -len(str(wf[k]['inputs'].get('text', ''))))
    return (enc[0] if enc else None), (enc[1] if len(enc) > 1 else None)


def prepare(pipeline, prompt, w, h, seed, negative='', cfg=1.0, steps=4, bc_sua=4,
            sampler='euler', scheduler='simple', ten_file='flux/anh', batch=1):
    path = os.path.join(WORKFLOW_DIR, '%s.json' % pipeline)
    if not os.path.isfile(path):
        raise FileNotFoundError('%s không có — chạy Cell 4 trước' % path)
    wf = json.load(open(path, encoding='utf-8'))

    for nid in _nodes(wf, 'LoadImage'):
        name = wf[nid]['inputs'].get('image', '')
        if not os.path.isfile(os.path.join(IN_DIR, name)):
            raise FileNotFoundError(
                '❌ %s cần file %s/%s — hãy upload ảnh + mask vào đó, '
                'hoặc dùng Cell 7 (vẽ mask bằng chuột, tự upload).'
                % (pipeline, IN_DIR, name))

    pos, neg = _pos_neg(wf)
    if pos:
        wf[pos]['inputs']['text'] = prompt
    if neg:
        wf[neg]['inputs']['text'] = negative

    for nid in _nodes(wf, 'EmptyLatentImage'):
        wf[nid]['inputs']['width'] = int(w)
        wf[nid]['inputs']['height'] = int(h)
        if 'batch_size' in wf[nid]['inputs']:
            wf[nid]['inputs']['batch_size'] = int(batch)

    for i, nid in enumerate(sorted(_nodes(wf, 'KSampler'), key=int)):
        wf[nid]['inputs'].update(
            seed=int(seed) + i, steps=int(steps), cfg=float(cfg),
            sampler_name=sampler, scheduler=scheduler)
    # FaceDetailer chạy trên vùng crop nhỏ: bước riêng (BC_SUA_CHI_TIET), nhưng cfg
    # theo ô CFG để negative có tác dụng cả ở bước sửa mặt / sửa tay.
    for i, nid in enumerate(sorted(_nodes(wf, 'FaceDetailer'), key=int)):
        wf[nid]['inputs'].update(
            seed=int(seed) + 100 + i, steps=int(bc_sua), cfg=float(cfg),
            sampler_name=sampler, scheduler=scheduler)

    for nid in _nodes(wf, 'SaveImage'):
        wf[nid]['inputs']['filename_prefix'] = ten_file
    return wf


def run(wf, timeout=900):
    r = requests.post('%s/prompt' % COMFY, json={'prompt': wf}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError('ComfyUI từ chối prompt (HTTP %s):\n%s'
                           % (r.status_code, json.dumps(r.json(), ensure_ascii=False)[:1500]))
    pid = r.json()['prompt_id']
    print('📤 prompt_id=%s' % pid)
    t0 = time.time()
    while time.time() - t0 < timeout:
        time.sleep(2)
        try:
            h = requests.get('%s/history/%s' % (COMFY, pid), timeout=10).json()
        except Exception:
            continue
        if pid in h:
            st = h[pid].get('status', {})
            if st.get('status_str') == 'error':
                raise RuntimeError('ComfyUI báo lỗi khi chạy: %s'
                                   % json.dumps(st, ensure_ascii=False)[:1200])
            files = []
            for node_out in h[pid].get('outputs', {}).values():
                for im in node_out.get('images', []):
                    if im.get('type') != 'output':
                        continue
                    files.append(os.path.join(OUT, im.get('subfolder', ''), im['filename']))
            return [f for f in files if os.path.isfile(f)]
    raise TimeoutError('Quá %ds chưa xong — xem /content/comfyui.log' % timeout)


def tom_tat(cfg_hinh):
    print('=' * 68)
    for k, v in cfg_hinh.items():
        print('  %-14s %s' % (k, v))
    print('=' * 68)


def generate(prompt=PROMPT, preset=PRESET, pipeline=PIPELINE, size=SIZE,
             them=THEM_VAO_PROMPT, neg_mode=NEG_MODE, negative_tu_viet=NEGATIVE_PROMPT,
             cfg=CFG, tu_dong_cfg=TU_DONG_BAT_CFG, steps=STEPS, bc_sua=BC_SUA_CHI_TIET,
             sampler=SAMPLER, scheduler=SCHEDULER, seed=SEED, n=SO_ANH,
             ten_file=TEN_FILE, nhieu=NHIEU_PROMPT, show=True, luu_drive=LUU_VAO_DRIVE):
    """Tạo ảnh. Mọi ô ở trên đều có thể truyền đè khi gọi bằng code."""
    prompt, pipeline, preset_size, doc = apply_preset(preset, prompt, pipeline, them)
    w, h = chon_size(size, preset_size)
    negative = gop_negative(neg_mode, negative_tu_viet, doc)
    cfg, steps, note = xu_ly_cfg(negative, cfg, steps, tu_dong_cfg)
    kiem_tra(prompt, w, h, cfg, steps, negative)

    ds = [d.strip() for d in str(nhieu or '').splitlines() if d.strip()]
    ds_prompt = ds if ds else [prompt]

    tom_tat({
        'pipeline': pipeline,
        'kich_thuoc': '%dx%d' % (w, h),
        'cfg / steps': '%.1f / %d' % (cfg, steps),
        'sampler': '%s + %s' % (sampler, scheduler),
        'sua chi tiet': '%d bước' % int(bc_sua),
        'negative': (negative[:68] + '...') if len(negative) > 68 else (negative or '(không)'),
        'so anh': '%d prompt x %d' % (len(ds_prompt), int(n)),
    })
    if note:
        print('ℹ️ %s' % note)
    if int(n) > 2:
        print('⚠️ %d ảnh liên tiếp trên T4 16GB — nếu báo OOM, hạ xuống 1-2.' % int(n))

    if not _health():
        raise RuntimeError('❌ ComfyUI chưa chạy — chạy Cell 5 trước')

    ket_qua = []
    for pi, p_txt in enumerate(ds_prompt):
        for i in range(int(n)):
            s = int(seed) + i if int(seed) >= 0 else random.randint(0, 2 ** 31 - 1)
            t0 = time.time()
            wf = prepare(pipeline, p_txt, w, h, s, negative, cfg, steps, bc_sua,
                         sampler, scheduler, ten_file)
            files = run(wf)
            for f in files:
                ket_qua.append(f)
                if show:
                    display(Image.open(f))
            print('  [%d/%d] ảnh %d/%d: %d file, %.1fs, seed=%d'
                  % (pi + 1, len(ds_prompt), i + 1, int(n), len(files), time.time() - t0, s))

    try:
        with open('/content/lan_chay_cuoi.json', 'w', encoding='utf-8') as f:
            json.dump({'prompt': ds_prompt, 'negative': negative, 'cfg': cfg,
                       'steps': steps, 'bc_sua_chi_tiet': int(bc_sua),
                       'sampler': sampler, 'scheduler': scheduler, 'seed': seed,
                       'size': [w, h], 'pipeline': pipeline, 'preset': preset,
                       'anh': ket_qua}, f, ensure_ascii=False, indent=1)
    except Exception:
        pass

    if luu_drive and os.path.isdir('/content/drive/MyDrive'):
        import shutil
        dest = '/content/drive/MyDrive/FLUX_output'
        os.makedirs(dest, exist_ok=True)
        for f in ket_qua:
            shutil.copy2(f, dest)
        print('💾 Đã copy %d ảnh vào %s' % (len(ket_qua), dest))
    return ket_qua


def nhanh(prompt, n=1, **kw):
    """Một dòng lấy ảnh nhanh: pipeline fast, 1024x1024."""
    return generate(prompt=prompt, preset=TUY_CHON, pipeline='flux_q5_fast',
                    size='1024x1024 (vuông, ~1MP)', n=n, **kw)


def dep(prompt, n=1, **kw):
    """Một dòng lấy ảnh đẹp: quality, sửa cả mặt và tay."""
    return generate(prompt=prompt, preset=TUY_CHON, pipeline='flux_q5_quality',
                    size='832x1216 (dọc, ~1MP)', n=n, **kw)


anh = generate()
print()
print('📝 Prompt: ' + (PROMPT if len(PROMPT) <= 120 else PROMPT[:120] + '...'))
print('✅ %d ảnh trong %s' % (len(anh), OUT))
print('💡 Gọi lại nhanh: nhanh("prompt của bạn")  ·  dep("prompt của bạn", n=2)')


In [ ]:
# @title 🖌 CELL 7 — Inpaint vẽ tay (Gradio, dùng flux_q5_inpaint)
DENOISE = 0.5  # @param {type:"slider", min:0.2, max:0.85, step:0.05}
STEPS = 6  # @param {type:"integer"}
GROW_MASK = 12  # @param {type:"integer"}

import os, sys, json, time, uuid, random, subprocess, requests
import numpy as np
from PIL import Image

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'])
    import gradio as gr

sys.path.insert(0, '/content/workflows')
COMFY = 'http://127.0.0.1:8188'
OUT = '/content/ComfyUI/output'
WF_INPAINT = '/content/workflows/flux_q5_inpaint.json'

def latest_output():
    if not os.path.isdir(OUT):
        return None
    fs = [os.path.join(OUT, f) for f in os.listdir(OUT)
          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    return max(fs, key=os.path.getmtime) if fs else None

def split_bg_mask(value):
    """Tách (ảnh nền, mask) từ ImageEditor/Sketchpad/ImageMask của Gradio 3/4/5."""
    def to_img(x, mode):
        if isinstance(x, dict) and 'name' in x:
            return Image.open(x['name']).convert(mode)
        if isinstance(x, str) and os.path.isfile(x):
            return Image.open(x).convert(mode)
        if hasattr(x, 'convert'):
            return x.convert(mode)
        return Image.fromarray(np.array(x)).convert(mode)

    if isinstance(value, dict):
        if 'background' in value:                      # Gradio 5 ImageEditor
            bg = to_img(value['background'], 'RGB')
            layers = value.get('layers') or []
            if not layers:
                return bg, None
            alpha = to_img(layers[0], 'RGBA').split()[-1]
            return bg, alpha
        if 'image' in value and 'mask' in value:       # Gradio 3/4
            return to_img(value['image'], 'RGB'), to_img(value['mask'], 'L')
    if isinstance(value, (list, tuple)) and len(value) == 2:
        return to_img(value[0], 'RGB'), to_img(value[1], 'L')
    if value is not None and hasattr(value, 'convert'):
        return value.convert('RGB'), None
    return None, None

def upload(path):
    with open(path, 'rb') as f:
        r = requests.post(f'{COMFY}/upload/image', files={'image': f},
                          data={'overwrite': 'true', 'type': 'input', 'subfolder': ''},
                          timeout=60)
    r.raise_for_status()
    return r.json()['name']

def do_inpaint(editor, prompt, denoise, steps, grow, seed):
    if editor is None:
        latest = latest_output()
        if not latest:
            return None, '❌ Chưa có ảnh: upload ảnh hoặc chạy Cell 6 trước'
        bg, mask = Image.open(latest).convert('RGB'), None
    else:
        bg, mask = split_bg_mask(editor)
    if bg is None:
        return None, '❌ Không đọc được ảnh'
    if mask is None or float(np.mean(np.array(mask) > 128)) < 0.005:
        return None, '❌ Chưa tô mask — dùng cọ tô lên vùng cần sửa'
    if mask.size != bg.size:
        mask = mask.resize(bg.size)

    rid = uuid.uuid4().hex[:8]
    img_p, msk_p = f'/tmp/inp_{rid}.png', f'/tmp/msk_{rid}.png'
    bg.save(img_p)
    mask.convert('RGB').save(msk_p)
    img_name, msk_name = upload(img_p), upload(msk_p)

    wf = json.load(open(WF_INPAINT, encoding='utf-8'))
    wf['4']['inputs']['image'] = img_name
    wf['5m']['inputs']['image'] = msk_name
    wf['9e']['inputs']['grow_mask_by'] = int(grow)
    wf['7']['inputs']['denoise'] = float(denoise)
    wf['7']['inputs']['steps'] = int(steps)
    wf['7']['inputs']['seed'] = int(seed) if int(seed) >= 0 else random.randint(0, 2**31 - 1)
    if prompt.strip():
        wf['5']['inputs']['text'] = prompt.strip()

    r = requests.post(f'{COMFY}/prompt', json={'prompt': wf}, timeout=30)
    if r.status_code != 200:
        return None, f'❌ HTTP {r.status_code}: {r.text[:600]}'
    pid = r.json()['prompt_id']

    for i in range(300):
        time.sleep(2)
        try:
            h = requests.get(f'{COMFY}/history/{pid}', timeout=10).json()
        except Exception:
            continue
        if pid in h:
            for node_out in h[pid].get('outputs', {}).values():
                for im in node_out.get('images', []):
                    if im.get('type') != 'output':
                        continue
                    p = os.path.join(OUT, im.get('subfolder', ''), im['filename'])
                    if os.path.isfile(p):
                        return Image.open(p), f'✅ Xong — {os.path.basename(p)}'
            return None, '❌ Chạy xong nhưng không thấy ảnh — xem /content/comfyui.log'
    return None, '❌ Hết thời gian chờ'

default = latest_output()
with gr.Blocks(title='Inpaint FLUX Q5') as demo:
    gr.Markdown('## 🖌 Inpaint FLUX.1-schnell Q5\n'
                '1. Upload ảnh (để trống = lấy ảnh mới nhất trong output)\n'
                '2. **Tô lên vùng lỗi** (tay/mặt/chân) bằng cọ\n'
                '3. Mô tả phần muốn vẽ lại → bấm **Sửa vùng tô** (~20s trên T4)')
    with gr.Row():
        with gr.Column():
            ed = gr.ImageEditor(type='pil', height=620,
                                value={'background': Image.open(default).convert('RGB'),
                                       'layers': [], 'composite': Image.open(default).convert('RGB')}
                                if default else None,
                                brush=gr.Brush(colors=['#FFFFFF'], color_mode='fixed', default_size=40),
                                label='Ảnh gốc — tô lên vùng cần sửa')
            pr = gr.Textbox(label='Mô tả phần vẽ lại (tiếng Anh)',
                            value='detailed human hand, five fingers, natural fingernails, '
                                  'realistic skin texture, photorealistic, sharp focus', lines=2)
            with gr.Row():
                d = gr.Slider(0.2, 0.85, value=DENOISE, step=0.05, label='Denoise')
                st = gr.Slider(4, 12, value=STEPS, step=1, label='Steps')
            with gr.Row():
                g = gr.Slider(0, 32, value=GROW_MASK, step=2, label='Grow mask px')
                sd = gr.Number(value=-1, label='Seed (-1 = random)')
            btn = gr.Button('🖌 Sửa vùng tô', variant='primary')
        with gr.Column():
            out_img = gr.Image(label='Kết quả', height=620, type='pil')
            msg = gr.Markdown('Sẵn sàng.')
    btn.click(do_inpaint, inputs=[ed, pr, d, st, g, sd], outputs=[out_img, msg])

print('Đợi link https://xxxx.gradio.live (bấm Stop để tắt)')
demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860)


In [ ]:
# @title 🔎 CELL 8 — Chẩn đoán: node có đủ không, model có đủ không
import os, json, requests

COMFY = 'http://127.0.0.1:8188'
P = json.load(open('/content/mode_ai_paths.json'))

try:
    info = requests.get(f'{COMFY}/object_info', timeout=60).json()
except Exception as e:
    raise RuntimeError(f'❌ Không gọi được /object_info — ComfyUI chưa chạy? ({e})')

print(f'ComfyUI đang phục vụ {len(info)} node class\n')
CAN_CO = ['UnetLoaderGGUF', 'DualCLIPLoaderGGUF', 'VAELoader', 'KSampler', 'EmptyLatentImage',
          'CLIPTextEncode', 'VAEDecode', 'VAEEncode', 'ImageScaleBy', 'SaveImage', 'LoadImage',
          'ImageToMask', 'VAEEncodeForInpaint', 'FaceDetailer', 'SAMLoader',
          'UltralyticsDetectorProvider']
thieu = []
for n in CAN_CO:
    ok = n in info
    print(('  ✅ ' if ok else '  ❌ ') + n)
    if not ok:
        thieu.append(n)

print('\nModel ComfyUI nhìn thấy:')
for label, key in [('UNET GGUF', 'UnetLoaderGGUF'), ('CLIP', 'DualCLIPLoaderGGUF'),
                   ('VAE', 'VAELoader'), ('YOLO', 'UltralyticsDetectorProvider'),
                   ('SAM', 'SAMLoader')]:
    if key not in info:
        print(f'  {label}: (node thiếu)')
        continue
    node = info[key]
    seen = set()
    for field in ('unet_name', 'clip_name1', 'clip_name2', 'vae_name', 'model_name'):
        req = node.get('input', {}).get('required', {}).get(field)
        if isinstance(req, list) and req and isinstance(req[0], list):
            seen.update(req[0])
    print(f'  {label}: {sorted(seen) if seen else "(trống!)"}')

if 'UltralyticsDetectorProvider' in info:
    yolo = info['UltralyticsDetectorProvider']['input']['required']['model_name'][0]
    bad = [x for x in yolo if not (x.startswith('bbox/') or x.startswith('segm/'))]
    if yolo and bad == yolo:
        print('  ⚠️ YOLO không có tiền tố bbox/ — kiểm tra symlink models/ultralytics (Cell 2)')

print('\nVRAM:')
os.system('nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader')
if thieu:
    print('\n❌ THIẾU NODE: ' + ', '.join(thieu))
    print('   → GGUF thiếu: pip install "gguf>=0.13" sentencepiece protobuf')
    print('   → FaceDetailer/SAMLoader thiếu: pip install scikit-image piexif dill segment-anything')
    print('   → UltralyticsDetectorProvider thiếu: pip install ultralytics matplotlib')
    print('   Sau đó chạy lại Cell 2 rồi khởi động lại ComfyUI (Cell 5).')
else:
    print('\n✅ Đủ node cho cả 5 workflow')
print('\nLog ComfyUI (30 dòng cuối):')
os.system('tail -30 /content/comfyui.log')


In [ ]:
# @title 🌐 CELL 9 — Tunnel dự phòng (localtunnel) nếu cloudflared không chạy
import urllib.request, subprocess

subprocess.run(['npm', 'install', '-g', 'localtunnel'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=20).read().decode().strip()
print('🔑 Tunnel Password (dán khi trang hỏi):', ip)
subprocess.run(['lt', '--port', '8188'])


## ⚙️ Tham số tối ưu (đã đặt sẵn trong workflow)

| Nơi | Tham số | Giá trị | Vì sao |
|---|---|---|---|
| KSampler chính | steps | **4** | FLUX.1-schnell là model chưng cất 4 bước; thêm bước chỉ tốn thời gian |
| KSampler chính | cfg | **1.0** | schnell không dùng CFG → ComfyUI bỏ luôn nhánh negative (nhanh hơn) |
| KSampler chính | sampler / scheduler | `euler` / `simple` | bộ đôi ổn định nhất cho schnell |
| KSampler chính | size | 1024×1024 (hoặc 832×1216) | giữ tổng ~1 MP, bội số của 16 |
| FaceDetailer mặt | denoise / steps | 0.22 / 4 | đủ sửa mắt-miệng mà không đổi identity |
| FaceDetailer mặt | guide_size / max_size | 384 / 768 | crop nhỏ → nhanh, ít VRAM hơn 512/1024 |
| FaceDetailer mặt | SAM | `sam_vit_b` + threshold 0.93 | mask mặt sát, không lem |
| FaceDetailer tay | denoise / feather | 0.28 / 16 | không dùng SAM (tiết kiệm VRAM), feather lớn để blend |
| Hires | upscale 1.5× → denoise 0.35, 4 bước | — | không cần ESRGAN/UltimateSDUpscale, chỉ dùng node có sẵn |
| Inpaint | denoise 0.5, steps 6, grow_mask 12px | — | giữ context quanh vùng tô |
| ComfyUI | `--reserve-vram 1.0`, `--force-fp16`, `--fp16-vae`, SDPA | — | hợp với T4 16 GB (không dùng `--lowvram` vì ComfyUI mới đã có Dynamic VRAM) |

**Nếu OOM:** giảm `WIDTH/HEIGHT` về 832×832 · đặt `VAE_PREC=cpu-vae` · `PREVIEW=none` ·
`VRAM_MODE=lowvram` · tắt Cell 7 (Gradio) khi không dùng.

## 💡 Prompt — cách viết để KHÔNG bị lỗi

### Negative prompt vô dụng trên pipeline này

`comfy/samplers.py:610` — `if math.isclose(cond_scale, 1.0): uncond_ = None`.
Với cfg=1.0, ComfyUI **bỏ hẳn nhánh negative**. Viết "no extra fingers" vào cũng không được đọc.
Mọi "chống lỗi" phải nằm trong prompt **dương**.

### Tránh lỗi tay: đừng bắt model tự bịa ngón tay

| Viết cái này | Đừng viết |
|---|---|
| `hands tucked into pockets` | `five fingers` |
| `both hands wrapped around a ceramic cup` | `perfect hands` |
| `hands clasped together on her lap` | `detailed fingers` |
| `carrying a canvas tote bag` | (tay trôi nổi, không tả gì) |

Nghịch lý: càng nhấn mạnh số ngón, model chưng cất càng hay sinh **thêm** ngón.

### Cấu trúc prompt ăn với FLUX

**chủ thể → tư thế/tay → trang phục/bối cảnh → ánh sáng → ống kính → khung hình**

```
Close-up portrait of a young Vietnamese woman, natural skin with visible pores, soft window
light from the left, 85mm lens, shallow depth of field, head and shoulders framing, plain
warm backdrop, subtle film grain
```

Giữ ~1 megapixel (832×1216 / 1216×832 / 1024²) — xa khỏi ~1MP thì schnell sinh lỗi cấu trúc.

### Dùng preset có sẵn

Ô **PRESET** ở Cell 6 có 9 prompt kèm nhãn rủi ro: `chan_dung_can` (cận cảnh, không có tay),
`toan_than_tui_quan` (tay trong túi), `toan_than_ngoi` (tay đan) — rủi ro **Thấp**;
`ban_than_cam_coc`, `thoi_trang`, `duong_pho` — Trung bình; `anh_minh_hoa` — **Cao**
(YOLO mặt huấn luyện trên mặt người thật nên không nhận diện được mặt anime).

Nguồn: `scripts/prompt_presets.py` → `workflows/prompts.json`.

**Negative prompt có ở ô `NEG_MODE`** (9 chế độ: theo preset / chung / tay / mặt / chữ /
thừa chi / tất cả / tự viết / không dùng). Nhưng nhớ quy tắc `comfy/samplers.py:610`:
ở `cfg = 1.0` ComfyUI **bỏ hẳn nhánh negative**, viết vào cũng không được đọc.
Vì vậy Cell 6 in rõ trạng thái trước mỗi lần chạy — `✅ Negative đang BẬT (cfg=2.0 > 1.0)`
hoặc `⚠️ Có negative mà cfg=1.0 → negative KHÔNG được đọc` — và khi bạn bật negative nó
tự nâng `cfg 1.0 → 2.0`, `steps 4 → 8` để negative thật sự có tác dụng.
Muốn nhanh lại như cũ: `NEG_MODE = khong - không dùng negative`.

Các ô khác của Cell 6: `THEM_VAO_PROMPT` (nối thêm vào preset), `CFG`, `TU_DONG_BAT_CFG`,
`STEPS`, `BC_SUA_CHI_TIET` (bước riêng cho sửa mặt/tay), `SAMPLER`, `SCHEDULER`, `SEED`
(`-1` = ngẫu nhiên), `SO_ANH`, `SIZE` (6 khung hình ~1MP), `TEN_FILE`, `NHIEU_PROMPT`
(mỗi dòng một prompt), `LUU_VAO_DRIVE`. Gọi bằng code: `nhanh("...")` · `dep("...", n=2)`.

## 🧪 Tự kiểm tra (không cần GPU, không cần mạng)

```bash
python3 scripts/check_sync.py          # một lệnh làm hết
python3 scripts/check_sync.py --fix    # tự ghi lại artifact cho khớp nguồn
```

`check_sync.py` sinh lại toàn bộ artifact vào thư mục tạm, so **từng byte** với bản trong repo,
rồi chạy kiểm tra tĩnh. Cơ chế này có nghĩa là **bạn sửa workflow bằng cách sửa
`scripts/build_workflows.py`**, không phải sửa file JSON — nếu không, CI sẽ báo lệch.

Kiểm tra tĩnh đối chiếu từng node với `INPUT_TYPES`/`RETURN_TYPES` trích trực tiếp từ mã nguồn
ComfyUI + ComfyUI-GGUF + Impact Pack/Subpack (891 node class): thiếu input required, sai enum,
link đứt, sai kiểu dữ liệu, chu trình, sai tiền tố `bbox/` — bị bắt hết trước khi lên Colab.

Muốn cập nhật spec theo phiên bản ComfyUI/Impact Pack mới hơn:

```bash
python3 scripts/node_spec.py --comfy /tmp/ComfyUI --gguf /tmp/ComfyUI-GGUF         --impact /tmp/Impact-Pack --subpack /tmp/Impact-Subpack -o workflows/node_spec.json
```
